[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/29_adam.ipynb)

# 🟠 Medium: Adam Optimizer

Implement the **Adam** optimizer from scratch.

### Signature
```python
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8): ...
    def step(self): ...
    def zero_grad(self): ...
```

### Algorithm (per parameter)
```
m = β1 * m + (1-β1) * grad
v = β2 * v + (1-β2) * grad²
m̂ = m / (1 - β1ᵗ)    # bias correction
v̂ = v / (1 - β2ᵗ)
p -= lr * m̂ / (√v̂ + ε)
```

### My notes:
#### Why do `m` and `v` have the same shape as `w`?

Gradient-based optimizers operate on each individual parameter value using its corresponding gradient.

If:

```python
w.shape == (4, 3)
```

then:

```python
w.grad.shape == (4, 3)
```

because each element in `w` has its own gradient.

Optimizers that keep **per-parameter state** usually store that state with the same shape as the parameter tensor.

For Adam:

```python
m = torch.zeros_like(w)
v = torch.zeros_like(w)
```

so:

```python
m.shape == w.shape
v.shape == w.shape
```

Adam maintains:

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t
$$

$$
v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2
$$

where:

- `g` = gradient for each parameter value
- `m` = moving average of gradients for each parameter value
- `v` = moving average of squared gradients for each parameter value

Therefore:

```text
param, grad, m, v
```

all have the **same shape**.

A scalar `m` or `v` would mix together the gradient history of different parameter values.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [78]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        # pass  # store params, init m and v to zeros
        # self.params = [p for p in list(params) if p.requires_grad]
        self.params = list(params)
        # print(self.params)
        # print(len(self.params))
        
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        
        # self.step_count = 0
        self.step_count = [0] * len(self.params)

    def step(self):
        
        # pass  # update params using Adam rule
        
        # iterate through each layer
        for i, param in enumerate(self.params): 
            # i = 0 -> layer1.weight
            # i = 1 -> layer1.bias
            # i = 2 -> layer2.weight
            # i = 3 -> layer2.bias ...
                        
            # print(f"Param {i}: {param.data}, Grad: {param.grad}")
            if param.grad is None:
                continue
                
            self.step_count[i] += 1
            t = self.step_count[i]
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * param.grad
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * param.grad ** 2
            m_hat = self.m[i] / (1 - self.beta1**t)
            v_hat = self.v[i] / (1 - self.beta2**t)
            
            param.data -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        # pass  # zero all gradients
        for param in self.params:
            if param.grad is not None:
                # param.grad.detach_()
                param.grad.zero_()

In [79]:
# 🧪 Debug
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
opt = MyAdam([w, b], lr=0.01)
for i in range(5):
    # loss = (w ** 2).sum()
    loss = (w ** 2).sum() + (b ** 2).sum()
    loss.backward()
    print(w.grad)
    print(b.grad)
    opt.step()
    opt.zero_grad()
    print(w.grad)
    print(b.grad)
    print(f'Step {i}: loss={loss.item():.4f}')

tensor([[ 3.0820, -0.5869, -4.3576],
        [ 1.1369, -2.1690, -2.7972],
        [ 0.8067,  1.6761, -1.4385],
        [-0.8067, -1.1933,  0.3641]])
tensor([-1.7133,  2.2012, -2.1424])
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
tensor([0., 0., 0.])
Step 0: loss=15.6900
tensor([[ 3.0620, -0.5669, -4.3376],
        [ 1.1169, -2.1490, -2.7772],
        [ 0.7867,  1.6561, -1.4185],
        [-0.7867, -1.1733,  0.3441]])
tensor([-1.6933,  2.1812, -2.1224])
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
tensor([0., 0., 0.])
Step 1: loss=15.4268
tensor([[ 3.0420, -0.5469, -4.3176],
        [ 1.0969, -2.1290, -2.7572],
        [ 0.7667,  1.6361, -1.3985],
        [-0.7667, -1.1533,  0.3241]])
tensor([-1.6734,  2.1612, -2.1024])
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
tensor([0., 0., 0.])
Step 2: loss=15.1667
tensor([[ 3.0220, -0.5269, -4.2976],
        [ 1.076

In [59]:
# ✅ SUBMIT
from torch_judge import check
check('adam')


🧪 Testing: Adam Optimizer (Medium)
──────────────────────────────────────────────────
  ✅ [1/3] Parameters change after step (1.4ms)
  ✅ [2/3] Matches torch.optim.Adam (1.5ms)
  ✅ [3/3] zero_grad works (0.2ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (3.1ms total)
  Progress saved. Run status() to see your dashboard.

